# Robotics POC — Data Setup

This notebook:
1. Creates the Unity Catalog schema, volume, and Delta table
2. Generates realistic manometer gauge images and saves them to the volume
3. Inserts sample equipment reading data linked to the images

> **Run all cells top-to-bottom once.** Re-running is safe — it truncates and reloads data.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
CATALOG = "main"          # Change if you use a different Unity Catalog
SCHEMA  = "robotics_poc"
TABLE   = "equipment_readings"
VOLUME  = "manometer_images"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

print(f"Catalog : {CATALOG}")
print(f"Schema  : {CATALOG}.{SCHEMA}")
print(f"Table   : {CATALOG}.{SCHEMA}.{TABLE}")
print(f"Volume  : {VOLUME_PATH}")

In [ ]:
# ── Create Schema, Volume, and Delta Table ─────────────────────────────────
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✅  Schema  {CATALOG}.{SCHEMA} ready")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
print(f"✅  Volume  {CATALOG}.{SCHEMA}.{VOLUME} ready")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.{TABLE} (
  er_name              STRING  COMMENT 'Equipment / gauge identifier',
  er_value             DOUBLE  COMMENT 'AI-extracted reading value',
  er_angle             DOUBLE  COMMENT 'Manometer needle angle (0-240 deg sweep)',
  er_confidence_score  DOUBLE  COMMENT 'AI confidence score (0.0 – 1.0)',
  photo_volume_path    STRING  COMMENT 'Absolute path to manometer image in the volume',
  reviewer_value       DOUBLE  COMMENT 'Human-reviewed / corrected value',
  reviewer_comment     STRING  COMMENT 'Reviewer notes',
  mission_id           STRING  COMMENT 'Mission identifier',
  site                 STRING  COMMENT 'Inspection site name',
  date                 DATE    COMMENT 'Inspection date'
)
USING DELTA
COMMENT 'Equipment readings captured by robotic inspection missions'
""")
print(f"✅  Table   {CATALOG}.{SCHEMA}.{TABLE} ready")

In [ ]:
# ── Manometer Image Generator ──────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")          # non-interactive backend for cluster nodes
import matplotlib.pyplot as plt
import numpy as np
import os

def generate_manometer_image(value, max_val=100.0, unit="bar",
                              equipment_name="", save_path=""):
    """
    Draw a realistic analog manometer gauge and save it as a PNG.
    Gauge sweeps 240 degrees, with colour zones: green / amber / red.
    """
    fig, ax = plt.subplots(figsize=(5, 5))
    fig.patch.set_facecolor("#1a1a2e")
    ax.set_facecolor("#1a1a2e")
    ax.set_xlim(-1.65, 1.65)
    ax.set_ylim(-1.65, 1.65)
    ax.set_aspect("equal")
    ax.axis("off")

    # --- Outer bezel ---
    ax.add_patch(plt.Circle((0, 0), 1.50, color="#5d6d7e", zorder=1))
    # --- Gauge face ---
    ax.add_patch(plt.Circle((0, 0), 1.38, color="#f0f0f0", zorder=2))

    # --- Colour zones (240-degree sweep starting at 210°) ---
    START = 210.0   # degrees (math convention, CCW from +x)
    SWEEP = 240.0
    ZONES = [(0.00, 0.60, "#27ae60"),   # green
             (0.60, 0.80, "#f39c12"),   # amber
             (0.80, 1.00, "#c0392b")]   # red

    for pct_s, pct_e, colour in ZONES:
        t = np.linspace(np.radians(START - pct_s * SWEEP),
                        np.radians(START - pct_e * SWEEP), 80)
        rin, rout = 1.02, 1.24
        xs = np.concatenate([rout * np.cos(t), (rin * np.cos(t))[::-1]])
        ys = np.concatenate([rout * np.sin(t), (rin * np.sin(t))[::-1]])
        ax.fill(xs, ys, color=colour, alpha=0.88, zorder=3)

    # --- Major ticks and labels ---
    for i in range(11):
        pct = i / 10
        a = np.radians(START - pct * SWEEP)
        ax.plot([1.24 * np.cos(a), 1.36 * np.cos(a)],
                [1.24 * np.sin(a), 1.36 * np.sin(a)],
                color="#2c3e50", linewidth=2.5, zorder=4)
        if i % 2 == 0:
            lbl = f"{int(max_val * pct)}"
            ax.text(0.78 * np.cos(a), 0.78 * np.sin(a), lbl,
                    ha="center", va="center", fontsize=9,
                    fontweight="bold", color="#2c3e50", zorder=5)

    # --- Minor ticks ---
    for i in range(51):
        if i % 5 == 0:
            continue
        a = np.radians(START - (i / 50) * SWEEP)
        ax.plot([1.27 * np.cos(a), 1.36 * np.cos(a)],
                [1.27 * np.sin(a), 1.36 * np.sin(a)],
                color="#95a5a6", linewidth=0.8, zorder=4)

    # --- Needle ---
    pct_val = min(max(value / max_val, 0.0), 1.0)
    na = np.radians(START - pct_val * SWEEP)
    ax.plot([0, 0.96 * np.cos(na)], [0, 0.96 * np.sin(na)],
            color="#c0392b", linewidth=3.5, solid_capstyle="round", zorder=8)
    ax.plot([0, -0.22 * np.cos(na)], [0, -0.22 * np.sin(na)],
            color="#922b21", linewidth=5, solid_capstyle="round", zorder=8)

    # --- Centre cap ---
    ax.add_patch(plt.Circle((0, 0), 0.08, color="#2c3e50", zorder=9))
    ax.add_patch(plt.Circle((0, 0), 0.04, color="#ecf0f1", zorder=10))

    # --- Reading box ---
    ax.text(0, -0.52, f"{value:.1f} {unit}",
            ha="center", va="center", fontsize=13, fontweight="bold",
            color="#2c3e50", zorder=11,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor="#bdc3c7", linewidth=1.5))

    # --- Equipment label ---
    ax.text(0, 0.40, equipment_name, ha="center", va="center",
            fontsize=9, color="#7f8c8d", style="italic", zorder=11)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.tight_layout(pad=0.1)
    plt.savefig(save_path, dpi=100, bbox_inches="tight",
                facecolor="#1a1a2e", edgecolor="none")
    plt.close(fig)


# Quick test — display the generated image
test_path = f"{VOLUME_PATH}/test_manometer.png"
generate_manometer_image(72.5, max_val=160.0, unit="bar",
                         equipment_name="P1_Alpha", save_path=test_path)
print(f"✅  Test image saved: {test_path}")

from IPython.display import Image as IPImage, display as ipdisplay
ipdisplay(IPImage(test_path, width=280))

In [ ]:
# ── Generate Sample Data + Images ─────────────────────────────────────────
import pandas as pd
import random
from datetime import date, timedelta

random.seed(42)
np.random.seed(42)

# Site → equipment list,  missions,  (max_val, unit) per type prefix
SITE_CONFIG = {
    "Refinery Alpha": {
        "equipment": ["P1_Alpha", "P2_Alpha", "F1_Alpha", "T1_Alpha", "T2_Alpha"],
        "missions":  ["MSN-2026-001", "MSN-2026-002"],
        "specs": {"P": (160.0, "bar"), "F": (500.0, "L/min"), "T": (200.0, "°C")},
    },
    "Offshore Platform B": {
        "equipment": ["P1_Beta", "P2_Beta", "F1_Beta", "T1_Beta"],
        "missions":  ["MSN-2026-003", "MSN-2026-004"],
        "specs": {"P": (250.0, "bar"), "F": (300.0, "L/min"), "T": (150.0, "°C")},
    },
    "Pipeline Station C": {
        "equipment": ["P1_Gamma", "P2_Gamma", "F1_Gamma"],
        "missions":  ["MSN-2026-005"],
        "specs": {"P": (100.0, "bar"), "F": (800.0, "L/min"), "T": (120.0, "°C")},
    },
}

REVIEWER_COMMENTS = [
    "Reading confirmed accurate",
    "Slight variance — within tolerance",
    "AI reading accepted",
    "Manual check performed — value corrected",
    "Image quality good, reading reliable",
    "Re-inspection scheduled",
    "",
]

base_date = date(2026, 2, 1)
records   = []
rec_id    = 1

for site, cfg in SITE_CONFIG.items():
    for mission_id in cfg["missions"]:
        mission_date = base_date + timedelta(days=random.randint(0, 39))
        for eq_name in cfg["equipment"]:
            prefix              = eq_name[0]                         # P / F / T
            max_val, unit       = cfg["specs"].get(prefix, (100.0, "unit"))
            true_val            = round(random.uniform(max_val * 0.15, max_val * 0.92), 1)
            er_val              = round(true_val + np.random.normal(0, max_val * 0.018), 1)
            er_angle            = round((true_val / max_val) * 240.0, 2)
            confidence          = round(random.uniform(0.71, 0.99), 4)
            reviewer_val        = round(true_val + np.random.normal(0, max_val * 0.004), 1)
            reviewer_comment    = random.choice(REVIEWER_COMMENTS)

            img_filename = f"reading_{rec_id:03d}_{eq_name}.png"
            img_path     = f"{VOLUME_PATH}/{img_filename}"

            generate_manometer_image(
                value=er_val, max_val=max_val, unit=unit,
                equipment_name=eq_name, save_path=img_path
            )

            records.append({
                "er_name":             eq_name,
                "er_value":            er_val,
                "er_angle":            er_angle,
                "er_confidence_score": confidence,
                "photo_volume_path":   img_path,
                "reviewer_value":      reviewer_val,
                "reviewer_comment":    reviewer_comment,
                "mission_id":          mission_id,
                "site":                site,
                "date":                mission_date,
            })
            rec_id += 1

print(f"✅  Generated {len(records)} records and {rec_id - 1} manometer images")

In [ ]:
# ── Write to Delta Table ───────────────────────────────────────────────────
pdf = pd.DataFrame(records)

# Cast date column properly
pdf["date"] = pd.to_datetime(pdf["date"]).dt.date

sdf = spark.createDataFrame(pdf)

# Truncate first so re-runs don't duplicate rows
spark.sql(f"TRUNCATE TABLE {CATALOG}.{SCHEMA}.{TABLE}")
sdf.write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")

count = spark.sql(f"SELECT COUNT(*) AS n FROM {CATALOG}.{SCHEMA}.{TABLE}").collect()[0][0]
print(f"✅  Inserted {count} rows into {CATALOG}.{SCHEMA}.{TABLE}")

In [ ]:
# ── Verify ─────────────────────────────────────────────────────────────────
print("=== Sample rows ===")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.{TABLE} LIMIT 10"))

print("\n=== Row counts by site ===")
display(spark.sql(f"""
    SELECT site, COUNT(*) AS readings, ROUND(AVG(er_confidence_score), 3) AS avg_confidence
    FROM {CATALOG}.{SCHEMA}.{TABLE}
    GROUP BY site ORDER BY site
"""))

imgs = [f for f in os.listdir(VOLUME_PATH) if f.endswith(".png")]
print(f"\n=== Images in volume: {len(imgs)} ===")
for p in sorted(imgs)[:5]:
    print(f"  {VOLUME_PATH}/{p}")
print("  ...")